In [1]:
import os
import pandas as pd
import numpy as np
import json
import scipy.sparse as sp

In [2]:
RAW = "data/raw/online_retail_II.xlsx"
CACHE = "data/cache/raw_combined.pkl"
OUT = "data/processed"
os.makedirs("data/cache", exist_ok=True)
os.makedirs(OUT, exist_ok=True)

In [3]:
if os.path.exists(CACHE):
    df = pd.read_pickle(CACHE)
else:
    sheets = pd.read_excel(RAW, sheet_name=None, dtype={"Invoice": str, "StockCode": str})
    df = pd.concat(sheets.values(), ignore_index=True)
    df.to_pickle(CACHE)  

df.columns = df.columns.str.strip()
features = df.columns.tolist()
print(f"Total rows: {len(df)}")
print(f"Features: {', '.join(features)}")

Total rows: 1067371
Features: Invoice, StockCode, Description, Quantity, InvoiceDate, Price, Customer ID, Country


In [4]:
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [5]:
df.isna().describe()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
count,1067371,1067371,1067371,1067371,1067371,1067371,1067371,1067371
unique,1,1,2,1,1,1,2,1
top,False,False,False,False,False,False,False,False
freq,1067371,1067371,1062989,1067371,1067371,1067371,824364,1067371


In [6]:
df.isnull().sum()

Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

In [7]:
print(f"{float(df['Customer ID'].isna().sum()/df.shape[0] * 100):.4f}% \
of entries have no Customer ID; {df['Customer ID'].isna().sum()} rows")

22.7669% of entries have no Customer ID; 243007 rows


In [8]:
li = df[df['Quantity'] < 1].nunique().to_list()
returns = dict(zip(features, li))
print(f"There are {returns['Invoice']} invoices with negative quantity")

There are 11684 invoices with negative quantity


In [9]:
li2 = df[df['Price'] <= 0].nunique().to_list()
returns2 = dict(zip(features, li2))
print(f"There are {returns2['Invoice']} invoices with negative price")

There are 5384 invoices with negative price


In [10]:
print(f"There are {int(df['Invoice'].str.startswith('C', na=False).sum())} cancellations")

There are 19494 cancellations


In [11]:
df["StockCode"] = df["StockCode"].astype(str).str.strip().str.upper()

In [12]:
correct_StockCode = int(df['StockCode'].str.fullmatch(r'\d{5}[A-Z]*', na=False).sum())
print(f"{len(df)-correct_StockCode} non-products are present")

6093 non-products are present


In [13]:
print(f"{df.duplicated().sum()/df.shape[0] * 100:.4f}% \
of entries are duplicate rows; {df.duplicated().sum()} rows")

3.2168% of entries are duplicate rows; 34335 rows


In [14]:
initial_rows = len(df)
cleaning_log = []
def log_step(rule_name, before_df, after_df):
    dropped = len(before_df) - len(after_df)
    cleaning_log.append({
        "Cleaning Step": rule_name,
        "Rows Retained": len(after_df),
        "Rows Dropped": dropped,
        "% of Original Dropped": f"{(dropped / initial_rows) * 100:.2f}%"
    })
    return after_df
df = log_step("Drop missing Customer ID", df, df[df['Customer ID'].notna()])
df = log_step("Drop exact Duplicates", df, df.drop_duplicates(keep='first'))
df = log_step("Drop non product StockCode", df, df[df['StockCode'].str.fullmatch(r'\d{5}[A-Z]*$')])
df = log_step("Drop negative price", df, df[df['Price'] > 0])
df['Customer ID'] = df['Customer ID'].astype(int).astype(str)
common_description = (
    df.groupby(['StockCode', 'Description'])
      .size()
      .reset_index(name='count')
      .sort_values('count', ascending=False)
      .drop_duplicates(subset='StockCode')
      .set_index('StockCode')['Description']
)
df['Description'] = df['StockCode'].map(common_description)
cleaning_log_df = pd.DataFrame(cleaning_log)
print(f"\n Retained {len(df):,} out of {initial_rows:,} rows ({(len(df)/initial_rows)*100:.2f}%).")
cleaning_log_df


 Retained 794,163 out of 1,067,371 rows (74.40%).


,Cleaning Step,Rows Retained,Rows Dropped,% of Original Dropped
0,Drop missing Customer ID,824364,243007,22.77%
1,Drop exact Duplicates,797885,26479,2.48%
2,Drop non product StockCode,794223,3662,0.34%
3,Drop negative price,794163,60,0.01%


In [15]:
cutoff_date = df['InvoiceDate'].max() - pd.DateOffset(months=2)
print(cutoff_date)
def net(d, label):
    is_cancel = (d['Invoice'].str.startswith('C') | (d['Quantity'] <=0))
    pos = d.groupby(['Customer ID', 'StockCode'])['Quantity'].sum()
    returned = pos[pos <=0].index
    key = pd.MultiIndex.from_frame(d[['Customer ID', 'StockCode']])
    fully_returned = key.isin(returned) & ~is_cancel
    ac = log_step(f"{label} drop cancellations", d, d[~is_cancel])
    return log_step(f"{label} drop fully returned purchases", ac, d[~is_cancel & ~fully_returned])
train_df = net(df[df['InvoiceDate'] < cutoff_date].copy(), 'Tain')
test_df = net(df[df['InvoiceDate'] >= cutoff_date].copy(), 'Test')
cleaning_log_df = pd.DataFrame(cleaning_log)
cleaning_log_df

2011-10-09 12:50:00


,Cleaning Step,Rows Retained,Rows Dropped,% of Original Dropped
0,Drop missing Customer ID,824364,243007,22.77%
1,Drop exact Duplicates,797885,26479,2.48%
2,Drop non product StockCode,794223,3662,0.34%
3,Drop negative price,794163,60,0.01%
4,Tain drop cancellations,660194,15526,1.45%
5,Tain drop fully returned purchases,657296,2898,0.27%
6,Test drop cancellations,116383,2060,0.19%
7,Test drop fully returned purchases,115920,463,0.04%


In [16]:
min_customers = 10
train_pairs = train_df[["Customer ID", "StockCode"]].drop_duplicates()
n_customers = train_pairs.groupby('StockCode').size()
keep_products = n_customers[n_customers >= min_customers].index
pairs = train_pairs[train_pairs['StockCode'].isin(keep_products)]
print(f"Products Kept: {len(keep_products)} of {len(n_customers)}")
customers = np.sort(pairs["Customer ID"].unique())
products = np.sort(pairs["StockCode"].unique())
cust_idx = pd.Series(np.arange(len(customers)), index=customers)
prod_idx = pd.Series(np.arange(len(products)), index=products)
A = sp.csr_matrix((np.ones(len(pairs)),
                   (pairs["Customer ID"].map(cust_idx).to_numpy(), pairs["StockCode"].map(prod_idx).to_numpy())),
                  shape=(len(customers), len(products)))
print(A.shape)

Products Kept: 3680 of 4519
(5456, 3680)


In [17]:
test_pairs = test_df[['Customer ID', 'StockCode']].drop_duplicates()
test_pairs = test_pairs[test_pairs['Customer ID'].isin(cust_idx.index)]
m = test_pairs.merge(train_pairs, on=["Customer ID", "StockCode"], how="left", indicator=True)
heldout = m[m["_merge"] == "left_only"].drop(columns="_merge").copy()
heldout["cust_row"] = heldout["Customer ID"].map(cust_idx)
heldout["in_matrix"] = heldout["StockCode"].isin(prod_idx.index)
heldout["prod_col"] = heldout["StockCode"].map(prod_idx).astype("Int64")
print(f"Test customers: {heldout['cust_row'].nunique()}")
print(f"Held-out pairs: {len(heldout):,}  |  not in matrix: {(~heldout['in_matrix']).mean():.1%}")

Test customers: 1919
Held-out pairs: 49,214  |  not in matrix: 9.5%


In [18]:
country = (train_df.groupby(["Customer ID", "Country"]).size().reset_index(name="n")
           .sort_values("n", ascending=False).drop_duplicates("Customer ID")
           .set_index("Customer ID")["Country"])
sp.save_npz(f"{OUT}/A_train.npz", A)
pd.DataFrame({"col": prod_idx.values, "StockCode": prod_idx.index,
              "Description": prod_idx.index.map(common_description),
              "n_customers": prod_idx.index.map(n_customers)}).to_csv(f"{OUT}/products.csv", index=False)
pd.DataFrame({"row": cust_idx.values, "Customer ID": cust_idx.index,
              "Country": cust_idx.index.map(country)}).to_csv(f"{OUT}/customers_internal.csv", index=False)
heldout[["cust_row", "prod_col", "StockCode", "in_matrix"]].to_csv(f"{OUT}/heldout.csv", index=False)
cleaning_log_df.to_csv(f"{OUT}/cleaning_log.csv", index=False)

In [19]:
rows, cols = A.shape
report = {
    "customers": rows, "products": cols, "non_zeros": int(A.nnz),
    "sparsity_%": round(100 * (1 - A.nnz / (rows * cols)), 3),
    "dense_MB": round(rows * cols * 8 / 1e6, 1),
    "sparse_MB": round((A.data.nbytes + A.indices.nbytes + A.indptr.nbytes) / 1e6, 2),
    "cutoff": str(cutoff_date), "min_customers_per_product": min_customers,
    "test_customers": int(heldout["cust_row"].nunique()), "heldout_pairs": len(heldout),
    "heldout_not_in_matrix_%": round(100 * (~heldout["in_matrix"]).mean(), 2),
}
with open(f"{OUT}/matrix_report.json", "w") as f:
    json.dump(report, f, indent=2)
pd.Series(report)

customers                                   5456
products                                    3680
non_zeros                                 411136
sparsity_%                                97.952
dense_MB                                   160.6
sparse_MB                                   4.96
cutoff                       2011-10-09 12:50:00
min_customers_per_product                     10
test_customers                              1919
heldout_pairs                              49214
heldout_not_in_matrix_%                     9.47
dtype: object